# NumiNaMath Dataset Evaluation Notebook

This notebook demonstrates how to:
1. Load the NumiNaMath dataset from local files or HuggingFace.
2. Preprocess samples to extract the problem and boxed solution.
3. Call a frontier model API (e.g., GPT) using an API key from environment variables.
4. Run inference on 5 random test samples.
5. Compute cross-entropy between ground truth and prediction.


## 1. Import Required Libraries

We will use `datasets` for data loading, `re` for regex extraction, `os` for environment variables, and `random` for sampling.

In [1]:
import os
from pathlib import Path

#------------ CACHE ENVIRONMENT ----------------------------------------
# Set cache environment variables BEFORE importing datasets
# or cache env variablew will be reset at import and trigger errors
PROJECT_ROOT = Path.cwd().parent  # Go up one level
hf_cache = PROJECT_ROOT / ".cache" / "huggingface"
hf_cache.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(hf_cache)
os.environ["HF_DATASETS_CACHE"] = str(hf_cache / "datasets")
os.environ["HF_HUB_CACHE"] = str(hf_cache)
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache / "transformers")

# -------------- other imports -----------------------------------------
import re
import random
from datasets import load_dataset, Dataset, load_from_disk
import requests
import numpy as np

## 2. Load the NumiNaMath Dataset

Try to load the dataset from the local directory. If not found, download from HuggingFace.

In [2]:
# Define local dataset path
local_dataset_path = Path('../datasets/test/data-00000-of-00001.arrow').resolve()

# dataset ID
HF_DATASET_ID = "AI-MO/NuminaMath-TIR"
PROJECT_ROOT = Path.cwd().parent  # Go up one level
DATASET_DIR = PROJECT_ROOT / "datasets"

# logic
if local_dataset_path.exists():
    print(f"Loading dataset from local path: {local_dataset_path}")
    # dataset = Dataset.load_from_disk(str(local_dataset_path.parent.parent))
    dataset = load_from_disk(str(DATASET_DIR))
    test_split = dataset['test']
else:
    print("Local dataset not found. Downloading from HuggingFace...")
    dataset = load_dataset(HF_DATASET_ID)
    test_split = dataset['test']

print(f"Loaded {len(test_split)} test samples.")

Loading dataset from local path: /home/benjamin.deporte/GenAI_Math/datasets/test/data-00000-of-00001.arrow
Loaded 99 test samples.


In [3]:
dataset['test'][0]

{'problem': "In 1988, a person's age was equal to the sum of the digits of their birth year. How old was this person?",
 'solution': 'To solve this problem, let\'s break it down step-by-step:\n\n1. Let the person\'s birth year be \\( Y \\).\n2. In 1988, the person\'s age would be \\( 1988 - Y \\).\n3. The sum of the digits of \\( Y \\) should be equal to their age in 1988.\n\nTherefore, we need to find a year \\( Y \\) such that:\n\n\\[ 1988 - Y = \\text{sum of the digits of } Y \\]\n\nWe can solve this by iterating through possible values for \\( Y \\) and checking if the condition holds.\n\nLet\'s write a Python script to find the correct birth year \\( Y \\).\n```python\ndef digit_sum(year):\n    """Calculate the sum of the digits of a year."""\n    return sum(int(digit) for digit in str(year))\n\ndef find_birth_year():\nprint((    for year in range(1900, 1989):))  # Reasonable range given the\n```\n```output\nCell In[210], line 6\n    for year in range(1900, 1989):  # Reasonable ra

## 3. Preprocessing: Extract Problem and Boxed Solution

Define a function to extract the problem and the boxed solution (using `\\boxed{...}` regex) from each sample. Handle missing boxed solutions gracefully.

In [4]:
# first utility function,
# extract the string between the opening \boxed{ and the corresponding closing }
def extract_boxed(solution: str) -> str:
    """ 
    count the opening { and closing } to extract the right string
    """
    start = solution.find(r'\boxed{')
    if start == -1:
        # raise ValueError("Boxed solution not found in solution field.")*
        return ""
        
    # Move to the opening brace
    brace_pos = start + len(r'\boxed{') - 1
    depth = 0
    for i in range(brace_pos, len(solution)):
        if solution[i] == '{':
            depth += 1
        elif solution[i] == '}':
            depth -= 1
            if depth == 0:
                return solution[brace_pos + 1:i]  # content between the braces
    raise ValueError("Unmatched braces in \\boxed{} expression.")

## utility function

def extract_problem_and_boxed_solution(sample):
    """
    Utility function to extract problem, solution, and boxed solution from a sample in the test split of NuminaMath
    Inputs :
        - sample (dict, with keys:
            'problem' : str,
            'solution' : str,
            'messages' : list of dict,
        }
        the value sample['solution'] contains the chain of thoughts and the ultimate solution in \boxed{ .. }
    Returns:
        - problem (str) : the problem
        - solution (str) : the whole ground truth response
        - boxed_solution (str) : the extract between \boxed{ and }, if any, None otherwise
    """
    
    problem = sample.get('problem', None)
    solution = sample.get('solution', None)
    if problem is None or solution is None:
        raise ValueError("Sample missing 'problem' or 'solution' field.")
    
    # Match everything between \\boxed{ and the corresponding closing }
    boxed_solution = extract_boxed(solution)
    
    return problem, solution, boxed_solution

In [5]:
results = []
sampled_indices = random.sample(range(len(test_split)), 3)

for idx in sampled_indices:
    sample = test_split[idx]
    try:
        problem, solution, boxed_solution = extract_problem_and_boxed_solution(sample)
    except ValueError as e:
        print(f"Skipping sample {idx}: {e}")
        continue
    
    results.append({
        'problem': problem,
        'solution': solution,
        'ground_truth': boxed_solution,
    })
    
for i, res in enumerate(results):
    print(f"sample {i+1}", "-"*40)
    print(f"problem: {res['problem']}")
    print(f"full solution: {res['solution']}")
    print(f"ground truth: {res['ground_truth']}\n\n")

sample 1 ----------------------------------------
problem: Among the 10 natural numbers arranged in ascending order: 1, 4, 8, 10, 16, 19, 21, 25, 30, 43, how many groups of consecutive numbers have a sum that is a multiple of 11? Please explain your reasoning.
full solution: To solve the problem, we need to find groups of consecutive numbers from the given list whose sum is a multiple of 11. Here are the steps to solve the problem:

1. **Understand the problem:** We need to find the number of groups of consecutive numbers in the list whose sum is a multiple of 11.

2. **List of numbers:** The given list is:
   \[
   \text{numbers} = [1, 4, 8, 10, 16, 19, 21, 25, 30, 43]
   \]

3. **Iterate through the list:** We will iterate through the list and consider all possible groups of consecutive numbers.

4. **Sum of groups:** For each group of consecutive numbers, we will calculate the sum and check if it is a multiple of 11. If it is, we will count it.

5. **Count the valid groups:** We wil

## 4. Set Up Model API Access

Read the API key from environment variables and define a function to call the model API (e.g., OpenAI GPT).

In [6]:
# Example for OpenAI GPT API (adjust endpoint/model as needed)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
assert OPENAI_API_KEY, "OPENAI_API_KEY not found in environment variables."

API_URL = "https://api.openai.com/v1/chat/completions"
MODEL_NAME = "gpt-3.5-turbo"  # Change as needed

def call_api(problem, max_tokens=4096):
    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json"
    }
    prompt = (
        "Solve the following math problem. "
        "Enclose your reasoning in <think> and </think>."
        "ALWAYS write your final answer in <answer>...</answer>."
        "The final answer MUST be either a mathematical expression, or a number."
        "Simplify the answer to a number or an irreductible fraction when possible. For example, simplify 8 / 4 into 2, simplify 18 / 15 into 6 / 5."
        "If the answer is a mathematical expression or a fraction, write it in LateX format within <answer> and </answer>."
        "Return only these tags.\n\n"
        f"Problem: {problem}"
    )
    data = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": max_tokens,
        "temperature": 0.0,
    }
    
    response = requests.post(API_URL, headers=headers, json=data)
    response.raise_for_status()
    result = response.json()['choices'][0]['message']

    return result['content'].strip()

## 5. Run Inference on some Random Test Samples

Randomly select some samples, extract the problem and boxed solution, call the model, and collect predictions.

In [7]:
N_SAMPLES=10

results = []
sampled_indices = random.sample(range(len(test_split)), N_SAMPLES)

for idx in sampled_indices:
    sample = test_split[idx]
    try:
        problem, solution, boxed_solution = extract_problem_and_boxed_solution(sample)
    except ValueError as e:
        print(f"Skipping sample {idx}: {e}")
        continue
    try:
        full_answer = call_api(problem)
    except Exception as e:
        print(f"API call failed for sample {idx}: {e}")
        full_answer = None, None
        
    # extract the string with the answer tags
    match = re.search(r"<answer>(.*?)</answer>", full_answer, re.DOTALL)
    if not match:
        raise ValueError("No <answer> tags found in response.")
    prediction = match.group(1).strip()
        
    results.append({
        'problem': problem,
        'solution': solution,
        'ground_truth': boxed_solution,
        'full_answer': full_answer,
        'prediction': prediction
    })

for i, res in enumerate(results):
    print(f"Sample {i+1}:")
    print(f"Problem: {res['problem']}")
    print(f"Full solution: {res['solution'][-100:]}")
    print(f"Ground Truth: {res['ground_truth']}")
    print(f"Full answer: {res['full_answer']}")
    print(f"Prediction: {res['prediction']}")
    print('-'*40, "\n")

Sample 1:
Problem: Antoine owns a strawberry farm that supplies strawberries to his local bakeries. The first bakery needs 2 sacks, the second bakery needs 4 sacks, and the third bakery needs 12 sacks of strawberries per week. How many sacks of strawberries does he need to supply all the bakeries in 4 weeks?
Full solution: needs to supply \(\boxed{72}\) sacks of strawberries to all the bakeries over the course of 4 weeks.
Ground Truth: 72
Full answer: <think> 
To find out how many sacks of strawberries Antoine needs to supply all the bakeries in 4 weeks, we first need to calculate the total number of sacks needed per week by adding up the requirements of each bakery. Then, we can multiply this total by 4 to account for the 4 weeks. 
The total number of sacks needed per week is 2 + 4 + 12 = 18 sacks. 
Multiplying this by 4 gives us 18 * 4 = 72 sacks. 
Therefore, Antoine needs 72 sacks of strawberries to supply all the bakeries in 4 weeks. 
</think>
<answer>72</answer>
Prediction: 72
---

# Comparing ground truth and predictions

In [8]:
for i, res in enumerate(results):
    print(f"Sample {i+1}:")
    print(f"Ground Truth: {res['ground_truth']}")
    print(f"Prediction: {res['prediction']}")
    print('-'*40, "\n")

Sample 1:
Ground Truth: 72
Prediction: 72
---------------------------------------- 

Sample 2:
Ground Truth: 7
Prediction: 1
---------------------------------------- 

Sample 3:
Ground Truth: 0.124
Prediction: 0.1245099
---------------------------------------- 

Sample 4:
Ground Truth: 18
Prediction: \[ y = \begin{cases} 
3x & \text{if } x \leq 6 \\
-12 + 5x & \text{if } 6 < x \leq 15 \\
-87 + 10x & \text{if } x > 15 
\end{cases} \]
---------------------------------------- 

Sample 5:
Ground Truth: 25
Prediction: $25
---------------------------------------- 

Sample 6:
Ground Truth: 17
Prediction: 17
---------------------------------------- 

Sample 7:
Ground Truth: 130
Prediction: The sum of all positive integers less than \( 10^6 \) which can be expressed as \( m! + n! \) is 443839.
---------------------------------------- 

Sample 8:
Ground Truth: 15
Prediction: 60
---------------------------------------- 

Sample 9:
Ground Truth: -2
Prediction: 1
-----------------------------------

In [9]:
import os
print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))
print("TRANSFORMERS_CACHE:", os.environ.get("TRANSFORMERS_CACHE"))

HF_HOME: /home/benjamin.deporte/GenAI_Math/.cache/huggingface
HF_HUB_CACHE: /home/benjamin.deporte/GenAI_Math/.cache/huggingface
TRANSFORMERS_CACHE: /home/benjamin.deporte/GenAI_Math/.cache/huggingface/transformers


In [10]:
# First attempt : use BERTScore

from bert_score import score

for i, res in enumerate(results):
    print(f"Sample {i+1}:")
    gt = res['ground_truth']
    pred = res['prediction']
    print(f"Ground Truth: {gt}")
    print(f"Prediction: {pred}")
    P, R, F1 = score([gt], [pred], lang="en")
    print(f"P = {P.item():.3f}, R = {R.item():.3f}, F1 = {F1.item():.3f}")
    print('-'*40, "\n")

Sample 1:
Ground Truth: 72
Prediction: 72


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


P = 1.000, R = 1.000, F1 = 1.000
---------------------------------------- 

Sample 2:
Ground Truth: 7
Prediction: 1


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


P = 0.912, R = 0.948, F1 = 0.930
---------------------------------------- 

Sample 10:
Ground Truth: 312
Prediction: 312


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


P = 1.000, R = 1.000, F1 = 1.000
---------------------------------------- 

